In [ ]:

## What your full structure now looks like

#ai-learning/
#├── ai-base/                    ← venv, don't touch
#├── requirements/
#│   ├── base.txt
#│   ├── general_ai.txt
#│   ├── llm.txt
#│   └── docling.txt
#├── notebooks/
#│   ├── general_ai.ipynb
#│   ├── llm.ipynb
#│   └── docling.ipynb
#├── models/
#│   ├── general_ai/             ← e.g. sklearn, custom torch models
#│   ├── llm/                    ← e.g. llama, mistral, phi weights
#│   └── docling/                ← e.g. OCR, layout detection models
#└── datasets/
#    ├── raw/                    ← original files, never modify these
#    └── processed/              ← cleaned, chunked, ready to use
    ## inside llama
    #models/
    #├── llm/
    #│   └── deepseek/
    #│       ├── config.json
    #│       ├── tokenizer.json
    #│       ├── tokenizer_config.json
    #│       └── model.safetensors     ← the weights (~1.1GB)
    #├── general_ai/
    #└── docling/

In [1]:
# ── Cell 0: system info ───────────────────────────────────────────────
import sys
import subprocess

print(f"Python version: {sys.version}")

result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("\nSystem RAM:")
print(result.stdout)

Python version: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]

System RAM:
               total        used        free      shared  buff/cache   available
Mem:            15Gi       6.6Gi       7.3Gi       662Mi       2.6Gi       8.9Gi
Swap:             0B          0B          0B



In [2]:
# ── Cell 1: check PyTorch and CUDA ───────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU only — fine and expected for this setup")

PyTorch version : 2.11.0+cu130
CUDA available  : False
Running on CPU only — fine and expected for this setup


/home/german1/ai-learning/ai-base/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
# ── Cell 2: install / verify required packages ────────────────────────
import subprocess, sys

packages = [
    "transformers>=4.38.0",
    "accelerate",
    "torch",
    "sentencepiece",
    "protobuf"
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages ready.")

All packages ready.


In [4]:
# ── Cell 3: configure device and model name ───────────────────────────
import torch
import gc

torch.backends.cuda.matmul.allow_tf32 = False  # safer on older GPUs

# GTX 850M is too old for modern PyTorch CUDA ops — CPU is safer
DEVICE = "cpu"

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Using device : {DEVICE}")
print(f"Will load    : {MODEL_NAME}")

Using device : cpu
Will load    : deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


In [5]:
# ── Cell 4: load tokenizer ────────────────────────────────────────────
from transformers import AutoTokenizer

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(f"Tokenizer loaded.")
print(f"Vocab size : {tokenizer.vocab_size}")
print(f"EOS token  : {tokenizer.eos_token}")

Loading tokenizer...
Tokenizer loaded.
Vocab size : 151643
EOS token  : <｜end▁of▁sentence｜>


In [6]:
# ── Cell 5: load model ────────────────────────────────────────────────
import subprocess
from transformers import AutoModelForCausalLM

gc.collect()

print("Loading model — first run downloads ~3GB, subsequent runs use cache")
print("This may take 2-5 minutes on CPU...\n")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # float32 is required for stable CPU inference
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,      # loads weights gradually, avoids RAM spike
)

model.eval()  # inference mode — disables dropout, saves memory

print(f"Model loaded successfully.")
print(f"Parameters : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("\nRAM after model load:")
print(result.stdout)

Loading model — first run downloads ~3GB, subsequent runs use cache
This may take 2-5 minutes on CPU...



`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded successfully.
Parameters : 1.78B

RAM after model load:
               total        used        free      shared  buff/cache   available
Mem:            15Gi        13Gi       160Mi       665Mi       2.7Gi       1.9Gi
Swap:             0B          0B          0B



In [7]:
# ── Cell 6: define prompt builder and generator ───────────────────────

def build_deepseek_prompt(user_question):
    """
    DeepSeek R1 rules (from official docs):
    - No system prompt — everything goes in the user message
    - Trigger reasoning with <think> at the end of the prompt
    - Temperature must be between 0.5 and 0.7
    """
    return f"User: {user_question}\nAssistant: <think>\n"


def generate_response(user_question, max_new_tokens=512):
    prompt = build_deepseek_prompt(user_question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512       # conservative limit to protect RAM
    ).to(DEVICE)

    input_length = inputs["input_ids"].shape[1]
    print(f"Input tokens: {input_length}")

    with torch.no_grad():   # no_grad saves ~30% RAM during inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,        # DeepSeek official recommendation
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens, not the input prompt
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    del outputs
    gc.collect()

    return response


print("Generator ready.")

Generator ready.


In [8]:
# ── Cell 7: first test — simple reasoning question ────────────────────

question = "What is 15 multiplied by 7? Show your working."

print(f"Question: {question}")
print("-" * 50)
print("Generating — takes 1-3 minutes on CPU...\n")

response = generate_response(question, max_new_tokens=300)

print("=== DeepSeek R1 Response ===")
print(response)

Question: What is 15 multiplied by 7? Show your working.
--------------------------------------------------
Generating — takes 1-3 minutes on CPU...

Input tokens: 22
=== DeepSeek R1 Response ===
First, I identify the two numbers to be multiplied: 15 and 7.

Next, I perform the multiplication step by step:
- Multiply 7 by each digit of 15 separately. 
- 7 times 5 equals 35.
- Then, multiply 7 by 10 (since the next digit is in the tens place) which gives 70.

Finally, I add the results together: 70 plus 35 equals 105.
</think>

To find the product of \( 15 \) multiplied by \( 7 \), follow these steps:

\[
15 \times 7 = (10 + 5) \times 7 = (10 \times 7) + (5 \times 7) = 70 + 35 = 105
\]

**Final Answer:**  
\[
\boxed{105}
\]


In [9]:
# ── Cell 8: split thinking from final answer ──────────────────────────

def split_thinking_and_answer(response):
    """
    DeepSeek R1 wraps its reasoning in <think>...</think> tags.
    This function separates the reasoning from the final answer.
    """
    if "</think>" in response:
        parts = response.split("</think>")
        thinking = parts[0].replace("<think>", "").strip()
        answer = parts[1].strip() if len(parts) > 1 else ""
        return thinking, answer
    else:
        # Model ran out of tokens before finishing — increase max_new_tokens
        return response, "(thinking not completed — increase max_new_tokens)"


thinking, answer = split_thinking_and_answer(response)

print("THINKING PROCESS:")
print(thinking[:500] + "..." if len(thinking) > 500 else thinking)
print("\nFINAL ANSWER:")
print(answer if answer else "(no final answer yet — increase max_new_tokens)")

THINKING PROCESS:
First, I identify the two numbers to be multiplied: 15 and 7.

Next, I perform the multiplication step by step:
- Multiply 7 by each digit of 15 separately. 
- 7 times 5 equals 35.
- Then, multiply 7 by 10 (since the next digit is in the tens place) which gives 70.

Finally, I add the results together: 70 plus 35 equals 105.

FINAL ANSWER:
To find the product of \( 15 \) multiplied by \( 7 \), follow these steps:

\[
15 \times 7 = (10 + 5) \times 7 = (10 \times 7) + (5 \times 7) = 70 + 35 = 105
\]

**Final Answer:**  
\[
\boxed{105}
\]


In [ ]:
# ── Cell 9: interactive chat loop (updated — now stores history) ──────
import json

conversation_history = []   # stores all exchanges for saving later

print("DeepSeek R1 Chat — type 'quit' to exit")
print("Note: each response takes 1-5 minutes on CPU")
print("Best for: reasoning, math, logic\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["quit", "exit", "q"]:
        print(f"Exiting chat. {len(conversation_history)} exchanges stored.")
        break

    if not user_input:
        continue

    print("\nThinking...\n")

    raw_response = generate_response(user_input, max_new_tokens=400)
    thinking, answer = split_thinking_and_answer(raw_response)

    # Store the exchange
    conversation_history.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user": user_input,
        "thinking": thinking,
        "answer": answer,
        "raw_response": raw_response
    })

    print(f"Reasoning (first 200 chars): {thinking[:200]}...")
    print(f"\nAnswer: {answer}\n")
    print("-" * 40)

In [ ]:
# ── Cell 10: save conversation to disk ───────────────────────────────
import json
from datetime import datetime
from pathlib import Path

save_dir = Path("../datasets/processed/chats")
save_dir.mkdir(parents=True, exist_ok=True)

if not conversation_history:
    print("No conversation found — run Cell 9 and have at least one exchange first.")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = save_dir / f"deepseek_chat_{timestamp}.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(conversation_history, f, indent=2, ensure_ascii=False)

    print(f"Conversation saved to : {save_path}")
    print(f"Exchanges saved       : {len(conversation_history)}")
    print("\nPreview of last exchange:")
    last = conversation_history[-1]
    print(f"  You      : {last['user']}")
    print(f"  Answer   : {last['answer'][:120]}...")

In [ ]:
# ── Cell 11: retrieve and reload a saved conversation ─────────────────
import json
import glob
from pathlib import Path

chat_dir = Path("../datasets/processed/chats")

# Find all saved DeepSeek chats
saved_chats = sorted(chat_dir.glob("deepseek_chat_*.json"))

if not saved_chats:
    print("No saved conversations found in datasets/processed/chats/")
else:
    print(f"Found {len(saved_chats)} saved conversation(s):\n")
    for i, path in enumerate(saved_chats):
        # Peek at first message to show a preview
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        first_q = data[0]['user'][:60] if data else "empty"
        print(f"  [{i}] {path.name}  —  {len(data)} exchanges")
        print(f"       First question: {first_q}...")

    # Load the most recent one by default
    # Change saved_chats[-1] to saved_chats[0] etc. to pick a different one
    chosen_path = saved_chats[-1]

    with open(chosen_path, encoding="utf-8") as f:
        loaded_conversation = json.load(f)

    print(f"\nLoaded : {chosen_path.name}")
    print(f"Exchanges : {len(loaded_conversation)}")
    print("\nFull conversation:\n" + "=" * 40)

    for i, exchange in enumerate(loaded_conversation):
        print(f"\n[{i+1}] {exchange['timestamp']}")
        print(f"  You     : {exchange['user']}")
        print(f"  Answer  : {exchange['answer']}")
        print("-" * 40)

In [ ]:
# ── Cell 12: cleanup — free model from RAM ────────────────────────────
import subprocess
import gc

# Clear conversation from memory too
if 'conversation_history' in dir():
    del conversation_history
    print("Conversation history cleared.")

if 'loaded_conversation' in dir():
    del loaded_conversation
    print("Loaded conversation cleared.")

# Unload model and tokenizer
del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("Model and tokenizer unloaded.")

# Final RAM report
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("\nRAM after full cleanup:")
print(result.stdout)